In [0]:
# Silver Layer — Incremental MERGE INTO with Partitioning & Data Quality

## Design Decisions
- **`%run 0.config`** supplies all paths/names — zero hardcoded strings.
- **`MERGE INTO` (upsert)** replaces the destructive `DROP + CTAS` pattern; only changed/new rows are written.
- **Partitioned by `nationality` (drivers) and `ingestion_date` (results)** — `race_id` partitioning creates 1 000+ small files; `ingestion_date` is a better partition key with `ZORDER BY (race_id, driver_id)` for query pruning.
- **`forename` and `surname` kept as separate columns** (drivers) in addition to aggregated `name`; preserves surname-level filtering for BI tools.
- **NOT NULL constraints** enforce key column presence before data lands here.
- **Row-count and NULL validations** confirm successful transformation before the Gold layer runs.


In [0]:
%run ./0.config

In [0]:
# ── Step 1: Create Silver tables (DDL) if they don't yet exist ───────────────
# NOTE: DROP TABLE has been removed. Use MERGE INTO (Steps 2 & 3) for all
#       incremental loads. Schema evolution is handled by ALTER TABLE.

spark.sql(f"USE CATALOG {CATALOG_NAME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(SILVER_SCHEMA, 'drivers')} (
    driver_id      INT       NOT NULL,
    driver_ref     STRING    NOT NULL,
    number         INT,
    code           STRING,
    forename       STRING    NOT NULL,
    surname        STRING    NOT NULL,
    name           STRING    NOT NULL,
    nationality    STRING,
    dob            STRING,
    ingestion_date TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (nationality)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'silver'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq(SILVER_SCHEMA, 'results')} (
    result_id         INT     NOT NULL,
    race_id           INT     NOT NULL,
    driver_id         INT     NOT NULL,
    constructor_id    INT     NOT NULL,
    number            INT,
    grid              INT,
    position          INT,
    position_text     STRING,
    position_order    INT,
    points            DOUBLE,
    laps              INT,
    time              STRING,
    milliseconds      BIGINT,
    fastest_lap       INT,
    rank              INT,
    fastest_lap_time  STRING,
    fastest_lap_speed STRING,
    status_id         INT,
    ingestion_date    TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality'                    = 'silver'
)
""")


In [0]:
# ── Step 2: MERGE INTO silver.drivers (incremental upsert) ───────────────────

spark.sql(f"""
MERGE INTO {fq(SILVER_SCHEMA, 'drivers')} AS tgt
USING (
    SELECT
        CAST(driverId AS INT)                          AS driver_id,
        driverRef                                       AS driver_ref,
        CAST(number AS INT)                             AS number,
        code,
        name.forename                                   AS forename,
        name.surname                                    AS surname,
        concat(name.forename, ' ', name.surname)        AS name,
        nationality,
        dob,
        current_timestamp()                             AS ingestion_date
    FROM {fq(BRONZE_SCHEMA, 'drivers')}
    WHERE driverId IS NOT NULL
) AS src
ON tgt.driver_id = src.driver_id
WHEN MATCHED AND (
    tgt.driver_ref  <> src.driver_ref  OR
    tgt.forename    <> src.forename    OR
    tgt.surname     <> src.surname     OR
    tgt.nationality <> src.nationality OR
    tgt.code        <> src.code
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [0]:
# ── Step 3: MERGE INTO silver.results (incremental upsert) ───────────────────

spark.sql(f"""
MERGE INTO {fq(SILVER_SCHEMA, 'results')} AS tgt
USING (
    SELECT
        CAST(resultId      AS INT)    AS result_id,
        CAST(raceId        AS INT)    AS race_id,
        CAST(driverId      AS INT)    AS driver_id,
        CAST(constructorId AS INT)    AS constructor_id,
        CAST(number        AS INT)    AS number,
        CAST(grid          AS INT)    AS grid,
        CAST(position      AS INT)    AS position,
        positionText                  AS position_text,
        CAST(positionOrder AS INT)    AS position_order,
        CAST(points        AS DOUBLE) AS points,
        CAST(laps          AS INT)    AS laps,
        time,
        CAST(milliseconds  AS BIGINT) AS milliseconds,
        CAST(fastestLap    AS INT)    AS fastest_lap,
        CAST(rank          AS INT)    AS rank,
        fastestLapTime                AS fastest_lap_time,
        fastestLapSpeed               AS fastest_lap_speed,
        CAST(statusId      AS INT)    AS status_id,
        current_timestamp()           AS ingestion_date
    FROM {fq(BRONZE_SCHEMA, 'results')}
    WHERE resultId IS NOT NULL AND raceId IS NOT NULL
) AS src
ON tgt.result_id = src.result_id
WHEN MATCHED AND (
    tgt.position         <> src.position         OR
    tgt.points           <> src.points           OR
    tgt.laps             <> src.laps             OR
    tgt.milliseconds     <> src.milliseconds     OR
    tgt.fastest_lap_time <> src.fastest_lap_time OR
    tgt.status_id        <> src.status_id
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [0]:
# ── Step 4: Data Quality Validation ──────────────────────────────────────────

from pyspark.sql import functions as F

def validate_silver(schema: str, table: str, pk_col: str, bronze_table: str):
    silver_df = spark.table(fq(schema, table))
    bronze_df = spark.table(fq(BRONZE_SCHEMA, bronze_table))

    silver_count = silver_df.count()
    bronze_count = bronze_df.count()
    null_pk      = silver_df.filter(F.col(pk_col).isNull()).count()
    null_date    = silver_df.filter(F.col("ingestion_date").isNull()).count()

    assert silver_count > 0,  f"[DQ FAIL] {fq(schema, table)}: table is empty!"
    assert null_pk      == 0, f"[DQ FAIL] {fq(schema, table)}: {null_pk} NULL PKs in '{pk_col}'"
    assert null_date    == 0, f"[DQ FAIL] {fq(schema, table)}: {null_date} NULL ingestion_dates"

    # Row count can legitimately differ after MERGE deduplication — log as warning, not failure.
    if silver_count != bronze_count:
        print(f"[DQ WARN] {fq(schema, table)}: Bronze:{bronze_count:,} vs Silver:{silver_count:,} "
              f"— investigate duplicates or intentional filter logic.")
    else:
        print(f"[DQ PASS] {fq(schema, table)}: {silver_count:,} rows | 0 NULL PKs | row count matches Bronze")

validate_silver(SILVER_SCHEMA, "drivers", "driver_id", "drivers")
validate_silver(SILVER_SCHEMA, "results", "result_id", "results")


In [ ]:
# ── Step 5: Maintenance — OPTIMIZE + ZORDER ──────────────────────────────────
# ZORDER co-locates related data in the same files, improving predicate push-down
# on the most common filter columns without creating per-value partitions.

spark.sql(f"OPTIMIZE {fq(SILVER_SCHEMA, 'drivers')} ZORDER BY (driver_id)")
spark.sql(f"OPTIMIZE {fq(SILVER_SCHEMA, 'results')} ZORDER BY (race_id, driver_id)")
print("OPTIMIZE complete for Silver layer.")

# Schedule VACUUM separately on a maintenance job (not inline with every pipeline run).
# spark.sql(f"VACUUM {fq(SILVER_SCHEMA, 'drivers')} RETAIN 168 HOURS")
# spark.sql(f"VACUUM {fq(SILVER_SCHEMA, 'results')} RETAIN 168 HOURS")
